### Api draft for outpu cu-project agent

This is the draft of how the api can look like. There is things that can be added and removed.  
This is also the output datamodel for how the agent will give a respons and is used in a validation step before it gets sent. So the output will always ahve this structure. 


With the RAG response, we can add:
- Excerpts: short descriptions of the event, places, or guide from Göteborg&CO dataset.
- Dates: for when events start and stop (a little unreliable not always specified correctly).
- Tags: for whether it is an event, guide, or place.
- Any other tag needed to make life easier.

If there needs to be a character limit for the input or the LLM response output (response_text), this needs to be specified here.

- ```AgentResponse``` is the top-level object returned by the agent.
- ```response_text``` contains the actual text answer to the user.
- ```results``` is a list of structured items from the RAG/tool layer, each with metadata like title, link, images, and optional location.
- ```meta``` contains optional metrics and logging data (confidence, latency, token usage).




In [ ]:
from pydantic import BaseModel, Field, HttpUrl

class Location(BaseModel):
    """Gives the location data."""
    name: str = Field(description="Location name")
    address: str | None = Field(description="Full address")
    lng_lat: dict[str] | None = Field(description="Longitude and latitude coordinates (optional)")

class ResultItem(BaseModel):
    """Response from the tool layer / RAG."""
    id: str = Field(description="Unique object ID")
    title: str = Field(description="Title of the source (article title, site, event, guide)")
    link: HttpUrl = Field(description="Link to source site")
    image_url_thumbnail: HttpUrl | None = Field(description="URL to thumbnail image connected to source")
    image_url_large: HttpUrl | None = Field(description="URL to large image connected to source")
    location: Location | None = Field(description="Location object containing address, name, and coordinates")

class ResponseMeta(BaseModel):
    """Optional logging and metadata for the response."""
    confidence: float = Field(description="Confidence score of the retrieved results")
    retrieval_source: str = Field(description="Origin/source of the retrieved information")
    tokens_used: int | None = Field(description="Number of tokens consumed for the response")
    latency_ms: int | None = Field(description="Time taken to retrieve/generate the response in milliseconds")

class AgentResponse(BaseModel):
    """Complete agent response output."""
    response_text: str = Field(description="Agent's answer to the user's question")
    results: list[ResultItem] = Field(description="Results retrieved from the tool layer / RAG")
    meta: ResponseMeta | None = Field(description="Optional metadata for logging or debugging")

```text
+----------------------+
|    User Prompt       |
+----------------------+
           |
           v
+----------------------+
|      LLM Agent       |
|  (Generates response |
|   and RAG query)     |
+----------------------+
           |
           v
+----------------------+
|  AgentResponse       |
|  (Top-level object)  |
+----------------------+
| response_text        | ---> "Answer to user"
| results [ResultItem] | ---> List of items from RAG/tool layer
| meta: ResponseMeta   | ---> Optional metrics (confidence, tokens, latency)
+----------------------+
           |
           v
+---------------------------+
|      ResultItem object    |
+---------------------------+
| id: str                   |
| title: str                |
| link: HttpUrl             |
| image_url_thumbnail: Url  |
| image_url_large: Url      |
| location: Location object |
+---------------------------+
           |
           v
+---------------------------+
|     Location object       |
+---------------------------+
| name: str                 |
| address: str | None       |
| lng_lat: dict | None      |
+---------------------------+